In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# RAG - PDF Search in multiple documents using Gemma 2 2B on Colab

This project demonstrates a pipeline for extracting, processing, and querying text data from PDF documents on Google Colab using natural language processing (NLP) techniques and Google's open-source model Gemma 2 2B. The system allows users to input a query, which is then answered based on the content of the PDFs.

## Setup
### Select the Colab runtime

To complete this tutorial, you'll need to have a Colab runtime with sufficient resources to run the Gemma model. In this case, you can use a T4 GPU:

1. In the upper-right of the Colab window, select **▾ (Additional connection options)**.
2. Select **Change runtime type**.
3. Under **Hardware accelerator**, select **T4 GPU**.

### Gemma 2 setup on Hugging Face

This cookbook uses Gemma 2B instruction tuned model through Hugging Face. So you will need to:

* Get access to Gemma 2 on [huggingface.co](huggingface.co) by accepting the Gemma 2 license on the Hugging Face page of the specific model, i.e., [Gemma 2B IT](https://huggingface.co/google/gemma-2-2b-it).
* Generate a [Hugging Face access token](https://huggingface.co/docs/hub/en/security-tokens) and configure it as a Colab secret 'HF_TOKEN'.

## Retrieval-Augmented Generation (RAG)

Large Language Models (LLMs) can learn new abilities without directly being trained on them. However, LLMs have been known to "hallucinate" when tasked with providing responses for questions they have not been trained on. This is partly because LLMs are unaware of events after training. It is also very difficult to trace the sources from which LLMs draw their responses from. For reliable, scalable applications, it is important that an LLM provides responses that are grounded in facts and is able to cite its information sources.

A common approach used to overcome these constraints is called Retrieval Augmented Generation (RAG), which augments the prompt sent to an LLM with relevant data retrieved from an external knowledge base through an Information Retrieval (IR) mechanism. The knowledge base can be your own corpora of documents, databases, or APIs.

## Installing and importing dependencies

In [ ]:
!pip install -q transformers sentence_transformers faiss-cpu torch PyPDF2 nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.0/27.0 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 15.6 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pandas as pd
import PyPDF2
import os
import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize
from google.colab import userdata

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


## Setting up the model and tokenizer

In [ ]:
HUGGING_FACE_ACCESS_TOKEN = userdata.get('HF_TOKEN')

model_name = 'google/gemma-2-2b-it'

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    token=HUGGING_FACE_ACCESS_TOKEN
    ).to('cuda')

tokenizer = AutoTokenizer.from_pretrained(model_name, token=HUGGING_FACE_ACCESS_TOKEN)

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

## Extracting and tokenizing info from the PDF files

The `extract_text_from_pdf()` function will look for all PDF files in the `pdf_path` folder.

The `split_text_into_chunks()` function gets the text and breaks it down into smaller chunks.

In [ ]:
def extract_text_from_pdf(pdf_path):
    try:
        with open(pdf_path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            text = "".join([page.extract_text() for page in reader.pages])
        return text
    except Exception as e:
        print(f"Error reading {pdf_path}: {e}")
        return ""

def split_text_into_chunks(text, max_chunk_size=1000):
    sentences = sent_tokenize(text)
    chunks = []
    current_chunk = ""

    for sentence in sentences:
        if len(current_chunk) + len(sentence) <= max_chunk_size:
            current_chunk += sentence + " "
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence + " "

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

## Extracting info from the PDFs

Set the variable `pdf_directory` with the path where your PDF files are. In this case, running on Google Colab, they would be in a folder called `PDFs`, that can be created on the content area on the left side.

A Pandas DataFrame is created containing the path of the corresponding PDF, its chunks and the embedding vector of its chunks.

In [ ]:
encoder = SentenceTransformer('all-MiniLM-L6-v2')

# Process PDF files
pdf_directory = "/content/drive/MyDrive/Data"
df_documents = pd.DataFrame(columns=['path', 'text_chunks', 'embeddings'])

for filename in os.listdir(pdf_directory):
    if filename.endswith(".pdf"):
        print(filename)
        pdf_path = os.path.join(pdf_directory, filename)
        text = extract_text_from_pdf(pdf_path)
        chunks = split_text_into_chunks(text)
        document_embeddings = encoder.encode(chunks)
        new_row = pd.DataFrame({'path': [pdf_path], 'text_chunks': [chunks], 'embeddings': [document_embeddings]})
        df_documents = pd.concat([df_documents, new_row], ignore_index=True)

df_documents

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

betel-vine-ipm-for-export.pdf
basil-ipm-for-export.pdf
bitter-gourd-ipm-for-export.pdf
bottle-gourd-ipm-for-export.pdf
brinjal-ipm-for-export.pdf
cowpea-ipm-for-export.pdf
custard-apple-ipm-for-export.pdf
drumstick-ipm-for-export.pdf
fenugreek-ipm-for-export.pdf
french-bean-ipm-for-export.pdf
AICRPAM at a Glance.pdf
lablab-bean-ipm-for-export.pdf
guava-ipm-for-export.pdf
little-gourd-ipm-for-export.pdf
mango-ipm-for-export.pdf
okra-ipm-for-export.pdf
pumpkin-ipm-for-export.pdf
ridge-gourd-ipm-for-export.pdf
sapota-ipm-for-export.pdf
snake-gourd-ipm-for-export.pdf
smooth-gourd-ipm-for-export.pdf
taro-ipm-for-export.pdf
an_overview_on_locust_control_amp_research.pdf
karnal_bunt-english.pdf
protocol_on_mass_production.pdf
sop_for_safe_and_judicious_use_of_tricyclazole_and_buprofezin_0.pdf
wheat_rust-english.pdf
standardoperatingproceduresforipm.pdf
opproc_IPM.pdf
sop_on_national_system_for_pest_monitoring_response_mechanism.pdf
major_pest_incidence_2015-16_to_2023-24.pdf
major_uses_of_pes

,path,text_chunks,embeddings
0,/content/drive/MyDrive/Data/betel-vine-ipm-for...,[II. Pest Surveillance \nWeekly monitoring th...,"[[0.0032100657, 0.068753906, -0.042986643, -0...."
1,/content/drive/MyDrive/Data/basil-ipm-for-expo...,[found free from insect pests then the field w...,"[[-0.035526168, 0.038986545, -0.030982926, -0...."
2,/content/drive/MyDrive/Data/bitter-gourd-ipm-f...,[II. Pest Surveillance \nWeekly monitoring th...,"[[0.038107026, 0.074034326, -0.003341412, -0.0..."
3,/content/drive/MyDrive/Data/bottle-gourd-ipm-f...,[Integrated Pest Management \n(IPM) in Bottle ...,"[[0.043319624, 0.017988123, -0.04401815, -0.01..."
4,/content/drive/MyDrive/Data/brinjal-ipm-for-ex...,[III. Integrated Pest Management strategies \...,"[[0.005716792, 0.08635328, 0.028907748, -0.055..."
5,/content/drive/MyDrive/Data/cowpea-ipm-for-exp...,[III. Integrated Pest Management strategies \...,"[[0.016463125, 0.12515272, -0.078511685, -0.05..."
6,/content/drive/MyDrive/Data/custard-apple-ipm-...,[III. Integrated Pest Management strategies \...,"[[0.0025834572, 0.012458311, -0.04669035, -0.0..."
7,/content/drive/MyDrive/Data/drumstick-ipm-for-...,[ Ploughing of orchard in November.  Avoid p...,"[[0.020837015, 0.021182034, -0.052014396, 0.00..."
8,/content/drive/MyDrive/Data/fenugreek-ipm-for-...,[III. Integrated Pe st Management strategies ...,"[[0.0029685323, 0.072004564, -0.041094415, -0...."
9,/content/drive/MyDrive/Data/french-bean-ipm-fo...,[II. Pest Surveillance \nWeekly monitoring th...,"[[0.034751672, 0.044572916, 0.027751619, -0.04..."


## Creating a FAISS index from all document embeddings

Faiss is a library for efficient similarity search and clustering of vectors. The IndexFlatL2 algorithm will be applied to all chunk embedding vectors.

In [ ]:
all_embeddings = np.vstack(df_documents['embeddings'].tolist())
dimension = all_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(all_embeddings)

## Calculating the embedding distance and generating an answer

The `find_most_similar_chunks()` function will create an embedding vector for your query and compare its similarity to all the chunks it retrieved from the PDF files, returning the most similar one, which will be used as the context for the next function.

The `generate_response()` function will generate an answer using our selected model (Gemma 2 2B) based on the context retrieved from the most similar info chunk.

In [ ]:
def find_most_similar_chunks(query, top_k=3):
    query_embedding = encoder.encode([query])
    distances, indices = index.search(query_embedding, top_k)
    results = []
    total_chunks = sum(len(chunks) for chunks in df_documents['text_chunks'])
    for i, idx in enumerate(indices[0]):
        if idx < total_chunks:
            doc_idx = 0
            chunk_idx = idx
            while chunk_idx >= len(df_documents['text_chunks'].iloc[doc_idx]):
                chunk_idx -= len(df_documents['text_chunks'].iloc[doc_idx])
                doc_idx += 1
            results.append({
                'document': df_documents['path'].iloc[doc_idx],
                'chunk': df_documents['text_chunks'].iloc[doc_idx][chunk_idx],
                'distance': distances[0][i]
            })
    return results

def generate_response(query, context, max_length=1000):
    prompt = f"Context: {context}\n\nQuestion: {query}\n\nAnswer:"
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to('cuda')

    with torch.no_grad():
        output = model.generate(input_ids, max_new_tokens=max_length, num_return_sequences=1)

    decoded_output = tokenizer.decode(output[0], skip_special_tokens=True)

    # Extracting the answer part by removing the prompt portion
    answer_start = decoded_output.find("Answer:") + len("Answer:")
    answer = decoded_output[answer_start:].strip()

    return answer

def query_documents(query):
    similar_chunks = find_most_similar_chunks(query)
    context = " ".join([result['chunk'].replace("\n", "") for result in similar_chunks])
    response = generate_response(query, context)
    return response, similar_chunks

## Looking for info in the PDFs

The variable `query` contains the information you want to retrieve from the PDF files.

In [ ]:
query = "Give me symptons causes and treatment for {tomato_early_blight?}"
answer, relevant_chunks = query_documents(query)

print(f"Query: {query}\n\n-----\n")
print(f"Generated answer: {answer}\n\n-----\n")
print("Relevant chunks:")
for chunk in relevant_chunks:
    print(f"Document: {chunk['document']}")
    print(f"Chunk: {chunk['chunk']}".replace("\n", ""))
    print(f"Distance: {chunk['distance']}")
    print()

Query: Give me symptons causes and treatment for tomato_early_blight?

-----

Generated answer: ## Tomato Early Blight: Symptoms, Causes, and Treatment

**Symptoms:**

* **Leaf Spots:**  Early blight starts with small, brown, irregular spots on the leaves. These spots often have a concentric ring pattern. As the disease progresses, the spots enlarge and become more necrotic (dead).
* **Leaf Necrosis:**  The affected leaves may turn yellow and die prematurely.
* **Stem Cankers:**  The disease can also infect the stems, causing cankers that can girdle the stem and lead to plant death.
* **Fruit Spots:**  Early blight can also infect the fruit, causing brown, sunken spots that can lead to fruit rot.

**Causes:**

* **Fungus:**  The primary cause of early blight is the fungus *Alternaria solani*. This fungus thrives in warm, humid conditions and can spread easily through wind and water.
* **Favorable Conditions:**  Early blight thrives in conditions with high humidity, temperatures between